# Untrained random policy

**Question:** What motion does a randomly initialized policy produce with small, bounded exploration?

Run this notebook to construct a fresh policy, record two seconds, and open its exact replay. No learning update occurs. The old coordinated wave is a visual reference only; its code is now Markdown below and cannot execute through Run All.

The policy receives joint angles and returns 18 offsets relative to neutral. Inputs are provisional; observation and reward design remain future work. `make_random_policy()` resets both random generators for a repeat.

In [ ]:
from pathlib import Path
import sys
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "spider" / "learning.py").is_file())
sys.path.insert(0, str(REPO_ROOT))
import mujoco
import numpy as np
import pandas as pd
from spider.learning import LearningSimulation, record_policy, plot_recordings
from spider.simulation import neutral_targets
sim = LearningSimulation()
print("Actuator order:", [sim.model.actuator(i).name for i in range(sim.model.nu)])
print("Timestep:", sim.model.opt.timestep, "s")

In [ ]:
# Fresh random policy: this does not use fixed_baseline or its wave.
# NumPy makes the forward pass explicit. There is no optimizer or learning here.
import numpy as np


def make_random_policy(weight_seed=7, action_seed=11):
    # Separate generators let us vary exploration while holding the network fixed.
    weight_rng = np.random.default_rng(weight_seed)
    action_rng = np.random.default_rng(action_seed)

    # A tiny neural network: 18 joint angles -> 32 hidden units -> 18 outputs.
    # Angles alone are a provisional input, not a complete description of motion:
    # this policy cannot directly observe joint velocity, body tilt, or contacts.
    # Each weight is drawn ONCE when we construct the policy, not at each step.
    w1 = weight_rng.normal(0.0, 1.0 / np.sqrt(18), size=(18, 32))
    b1 = np.zeros(32)
    w2 = weight_rng.normal(0.0, 0.01 / np.sqrt(32), size=(32, 18))
    b2 = np.zeros(18)

    max_offset_rad = 0.05  # Hard command bound: about 2.9 degrees from neutral.
    exploration_std = 0.20  # Noise scale BEFORE tanh; near zero, about 0.01 rad.

    def random_policy(observation):
        # Divide by 1 radian to state the input scale explicitly.
        x = np.asarray(observation.joint_positions, dtype=float) / 1.0
        hidden = np.tanh(x @ w1 + b1)
        mean = hidden @ w2 + b2  # Small initial mean; weights remain fixed.

        # Draw fresh, independent noise for all 18 joints on each policy call.
        # The policy has no prescribed leg coordination or temporal smoothing.
        noise = action_rng.normal(0.0, exploration_std, size=18)
        offsets = max_offset_rad * np.tanh(mean + noise)
        # tanh bounds commands smoothly. After tanh, they are NOT Gaussian.
        # LearningSimulation adds neutral targets and applies actuator limits.
        # Small angle bounds do not guarantee balance or hardware safety.
        return offsets

    return random_policy


random_policy = make_random_policy()
# Recreate with the same seeds before a repeat: calls advance the noise generator.


In [ ]:
# Fresh weights and exploration sequence for each repeat.
random_recording = record_policy(
    make_random_policy(weight_seed=7, action_seed=11),
    label="Untrained random | seeds 7/11 | bound 0.05 rad | noise 0.20",
    action_count=100, physics_steps=10,
)
# Validate the recording before presenting it as a two-second trial.
times = np.asarray(random_recording.replay.times)
expected = np.arange(101) * 10 * random_recording.replay.model.opt.timestep
assert np.allclose(times, expected, rtol=0, atol=1e-9), "Simulation clock reset: inspect invalid trial."
assert np.isfinite(random_recording.replay.states).all()
print(f"Recorded {times[-1]:.2f} seconds; {len(random_recording.offsets)} actions.")
fig, axes = plot_recordings(random_recording, actuator=0)


In [ ]:
# Replay saved states, rather than sampling another random trial.
# At the loop boundary the viewer jumps back to the initial pose.
viewer = random_recording.watch(REPO_ROOT / "telemetry" / "random-policy-replay", speed=1.0)


## Historical visual aid: coordinated motion (not executed)

The following preserves the earlier experiment and its dated observations. It is not the starting policy for learning.

# A fixed coordinated baseline

**Question:** Can a small, coordinated motion remain repeatable and numerically stable before we introduce learning?

Keep this controller fixed as a comparison for later policies. Initializing a learned policy near it is a separate choice. Later exploration could perturb amplitude, frequency, or relative phase instead of independently randomizing every actuator. That structure makes exploration easier to interpret, but also biases which motions it can discover.

We are not optimizing a loss, training a policy, or requiring a polished gait. Neutral standing remains a second control. The previous large-offset failure remains recorded in [02_first_policy](02_first_policy.ipynb).

**Robot connection:** [LearningSimulation.step](../../spider/learning.py) adds these radian offsets to the neutral targets, clips them to actuator limits, and advances the [same physics](../../spider/simulation.py). The existing [legacy gait](../../spider/controllers.py) is a separate treatment; this notebook does not replace it.

```python
from pathlib import Path
import sys
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "spider" / "learning.py").is_file())
sys.path.insert(0, str(REPO_ROOT))
import mujoco
import numpy as np
import pandas as pd
from spider.learning import LearningSimulation, record_policy, plot_recordings
from spider.simulation import neutral_targets
sim = LearningSimulation()
print("Actuator order:", [sim.model.actuator(i).name for i in range(sim.model.nu)])
print("Timestep:", sim.model.opt.timestep, "s")
```

## Read here: how the fixed baseline works

1. **One clock:** simulation time drives a sine wave at 0.5 Hz, so each cycle takes two seconds.
2. **Two groups:** front-left, middle-right, and rear-left share a phase. The other three legs are half a cycle behind. These are phase groups, not a guarantee that a particular tripod is supporting the body.
3. **Small joint offsets:** each leg receives coxa, hip, and knee amplitudes of 0.03, 0.015, and -0.02 radians—about 1.7°, 0.9°, and -1.1°. The knee moves opposite to the other two joints in local joint coordinates. This does not guarantee opposite world-space foot motion.
4. **Gentle startup:** a smooth ramp grows from zero to full amplitude over two seconds. The first command is neutral.
5. **No switching threshold:** the target changes smoothly. Unlike the previous policy, crossing a measured angle does not flip the target to the other extreme.

The policy uses time but no measured-state correction. The underlying position actuators still apply feedback to follow the commanded angles. Think of this as a motion pattern sent to those actuators, not a balance controller.

**Edit later:** `amplitudes`, `frequency_hz`, or `leg_phase`. First inspect the fixed run below. If you change a parameter, label the new treatment and keep the original baseline for comparison.

```python
amplitudes = np.array([0.03, 0.015, -0.02])  # coxa, hip, knee offsets in radians
frequency_hz = 0.5
startup_seconds = 2.0
# Leg order: front-left, front-right, middle-left, middle-right, rear-left, rear-right.
leg_phase = np.array([0, np.pi, np.pi, 0, 0, np.pi])

def fixed_baseline(observation):
    t = observation.time
    u = np.clip(t / startup_seconds, 0.0, 1.0)
    ramp = u * u * (3.0 - 2.0 * u)
    wave = np.sin(2.0 * np.pi * frequency_hz * t + leg_phase)
    leg_offsets = ramp * wave[:, None] * amplitudes[None, :]
    return leg_offsets.reshape(18)

def neutral_control(observation):
    return np.zeros(18)
```

## Run: neutral, coordinated baseline, exact repeat

Each trial starts from the same reset and lasts eight simulation seconds. Actions
update every 0.02 seconds. The repeat checks determinism, not robustness across
different starting states or disturbances. Check timestamps before interpreting
motion: a simulator reset makes the trial invalid.

The table reports displacement, height, body tilt and contacts. None is a learning
loss. The contact minimum uses the sampled 0.02-second endpoints; it can miss
shorter events between samples.

```python
# physics_steps = 10
# action_count = 400
# output_dir = REPO_ROOT / "telemetry" / "fixed-coordinated-baseline"
# output_dir.mkdir(parents=True, exist_ok=True)
# recordings = {}
# rows = []
# for label, controller in [("neutral", neutral_control), ("coordinated", fixed_baseline),
#                           ("repeat", fixed_baseline)]:
#     recording = record_policy(controller, label=label,
#                               physics_steps=physics_steps, action_count=action_count)
#     recordings[label] = recording
#     states = recording.measurements
#     times = np.asarray(recording.replay.times)
#     xyz = np.asarray([s.torso_position for s in states])
#     quat = np.asarray([s.torso_orientation for s in states])
#     tilt = np.degrees(np.arccos(np.clip(1 - 2 * (quat[:, 1]**2 + quat[:, 2]**2), -1, 1)))
#     expected_times = np.arange(action_count + 1) * physics_steps * recording.replay.model.opt.timestep
#     valid = bool(np.allclose(times, expected_times, rtol=0, atol=1e-9)
#                  and np.isfinite(recording.replay.states).all())
#     unclipped = np.asarray(neutral_targets()) + recording.offsets
#     rows.append({"treatment": label, "valid_timing_and_finite": valid,
#                  "dx_m": xyz[-1, 0] - xyz[0, 0], "dy_m": xyz[-1, 1] - xyz[0, 1],
#                  "min_height_m": xyz[:, 2].min(), "max_tilt_deg": tilt.max(),
#                  "minimum_contacts": min(len(s.foot_contacts) for s in states),
#                  "clipped_targets": int(np.count_nonzero(~np.isclose(unclipped, recording.targets)))})
#     run_dir = output_dir / label
#     run_dir.mkdir(exist_ok=True)
#     mujoco.mj_saveModel(recording.replay.model, str(run_dir / "model.mjb"))
#     np.savez(run_dir / "states.npz", states=np.asarray(recording.replay.states),
#              times=times, label=recording.replay.label, actuator_name="front_left_coxa_motor")
#     np.savez(run_dir / "measurements.npz", torso_position=xyz, tilt_degrees=tilt,
#              offsets=recording.offsets, targets=recording.targets, times=times)

# results = pd.DataFrame(rows).set_index("treatment")
# display(results)
# results.to_csv(output_dir / "summary.csv")
# repeat_identical = np.array_equal(recordings["coordinated"].replay.states,
#                                   recordings["repeat"].replay.states)
# print("Repeated coordinated states exactly identical:", repeat_identical)
# assert results.valid_timing_and_finite.all(), "Preserve and inspect an invalid trial before interpreting it."
# assert repeat_identical, "The supposedly fixed baseline did not repeat exactly."
# fig, axes = plot_recordings(recordings["neutral"], recordings["coordinated"], actuator=0)
# fig.savefig(output_dir / "comparison.png", dpi=150)
```

## Replay and inspect

This viewer replays the coordinated trial's recorded states. It does not rerun
the policy. Look for motion shared across legs and compare it with the target
curve. The repeating eight-second replay returns to its initial pose at the end;
that replay boundary is not a physical movement.

An eight-second stable trial is a bounded baseline, not evidence of walking,
disturbance recovery, or learning convergence.

**Recorded result:** All three trials completed eight seconds with valid timestamps and finite states. The coordinated repeat matched exactly. Maximum body tilt was 0.218 degrees, minimum torso height was 0.4495 m, sampled contacts ranged from three to six, and no targets clipped. Net horizontal displacement was approximately 1.7 mm. This supports a repeatable small motion at these settings; it does not establish locomotion or robustness.

```python
viewer = recordings["coordinated"].watch(REPO_ROOT / "telemetry" / "policy-replays", speed=1.0)
```

